In [ ]:
import random
from datasets import load_dataset
import os
import time

# ===============================================================================
# 🎓 [AI 코딩 튜터링] 데이터셋 분석 & 실습 예제 📚
# ===============================================================================
# ✨ 데이터셋 제목: yzhuang/mmlu_test_Korean_by_Meta-Llama-3-8B-Instruct
# ✨ 데이터셋 의미: 한국어 MMLU(Massive Multitask Language Understanding) 평가를 위한
# ✨ 데이터셋 설명: 여러 분야에 걸친 객관식 문항(Multiple Choice Questions)으로 구성되어 있습니다.
# 💡 목표: 이 데이터셋을 사용하여 주어진 질문과 보기만 가지고 AI가 정답을 추론하는 시뮬레이션을 해봅니다.
# -------------------------------------------------------------------------------

# 🚀 설정 값 (튜터가 적절히 조절할 수 있습니다!)
DATASET_NAME = "yzhuang/mmlu_test_Korean_by_Meta-Llama-3-8B-Instruct"
SAMPLE_COUNT = 10  # 전체 데이터셋 중 앞에서부터 몇 개의 샘플을 볼지 결정합니다. (튜터 모드에서는 적게 시작!)
# ===============================================================================


def load_data_with_robustness(dataset_id: str, split_name: str, sample_limit: int):
    """
    데이터셋을 로드하는 함수입니다. 스트리밍 실패에 대비한 안전장치를 포함합니다.
    """
    print("🤖 데이터셋 로딩을 시도합니다... (스트리밍 모드 우선)")
    dataset = None
    
    try:
        # 💡 1차 시도: 스트리밍(streaming=True)으로 로드!
        # 메모리 폭탄을 막고, 데이터셋을 아주 조금씩, 느리지만 안전하게 불러오는 방식입니다.
        dataset = load_dataset(dataset_id, split=split_name, streaming=True)
        print(f"✅ 성공: {split_name} 스플릿을 스트리밍 방식으로 성공적으로 로드했습니다.")
        
    except Exception as e:
        # ⚠️ 스트리밍이 불가능할 때의 대체 장치 (에러 핸들링)
        print(f"\n⚠️ 경고: 스트리밍 로드 실패! ({e}) -> 일반(Non-streaming) 모드로 전환합니다.")
        try:
            # 💡 2차 시도: 일반 모드로 로드하고, 혹시 모를 에러를 대비해 테스트 스플릿을 사용합니다.
            # 실제 교육 과정에서는 'train' 스플릿 전체를 가져오는 것이 좋지만, 예제 진행을 위해 테스트 스플릿으로 대체합니다.
            dataset = load_dataset(dataset_id, split=split_name)
            print(f"✅ 대체 성공: {split_name} 스플릿을 일반 모드로 로드했습니다.")
        except Exception as e_fallback:
            print(f"❌ 치명적 에러: 데이터셋 로딩에 실패했습니다. 에러: {e_fallback}")
            return None

    # ✨ 샘플링: 메모리 절약과 빠른 실습을 위해 전체 데이터셋이 아닌 일부만 가져옵니다.
    # streaming 모드와 일반 모드 모두 .take() 메서드를 지원합니다!
    sampled_dataset_iterator = dataset.take(sample_limit)
    
    return sampled_dataset_iterator


def run_mlmu_quiz_analysis(dataset_iterator):
    """
    데이터셋 이터레이터를 순회하며, AI 추론 시뮬레이션 실습을 수행합니다.
    """
    print("\n\n===============================================================================")
    print("🧠 MMLU 퀴즈 데이터셋 분석 및 AI 추론 시뮬레이션 시작!")
    print(f"🏃💨 총 {SAMPLE_COUNT}개의 샘플을 순차적으로 검사합니다.")
    print("===============================================================================")

    # 📝 파이썬 코딩 튜터 모드: 이터레이터(Iterator)를 사용해 순차적으로 데이터를 뽑아냅니다.
    sample_iterator = iter(dataset_iterator)
    
    # 첫 번째 샘플을 가져와서 루프를 돌립니다.
    for i, sample in enumerate(sample_iterator):
        print(f"\n{'='*50}\n[❓ {i+1}번째 퀴즈 분석 시작] 🧠")
        
        # 1. 핵심 정보 추출 및 출력
        subject = sample.get('subject', 'N/A')
        question = sample.get('question', 'N/A')
        answer_index = sample.get('answer', -1)
        choices_seq = sample.get('choices', [])
        
        # 2. 데이터 구조 분석 및 전처리 (Beginner Friendly)
        print(f"▶️ [주제] : {subject}")
        print(f"▶️ [질문] : {question[:50]}...") # 질문이 길 수 있으니 일부만 보여줍니다.
        
        # 3. 선택지(Choices) 가공: sequence 타입이므로 리스트로 깔끔하게 정리합니다.
        choices_list = choices_seq if isinstance(choices_seq, list) else [str(choices_seq)]
        print("\n--- 📚 문제 보기 (Choices) ---")
        for k, choice in enumerate(choices_list):
            # 보기 순서와 내용을 사용자 친화적으로 출력합니다.
            print(f"    [{chr(65+k)}] {choice}")
        print("---------------------------------")

        # 4. 정답 정보 확인 (답안지 열람!)
        # 이 데이터를 통해 우리가 "정답"이 무엇인지 파악할 수 있습니다.
        if answer_index is not None and answer_index >= 0:
            # 인덱스를 문자로 변환하여 정답을 알려줍니다.
            correct_answer_char = chr(65 + answer_index)
            print(f"\n[✅ 정답 확인] : 정답은 [{correct_answer_char}] 입니다. (실제 AI가 찾는 목표 지점!)")
        else:
            print("\n[⚠️ 경고] : 정답 정보를 찾을 수 없습니다. (데이터셋 구조를 확인해 보세요.)")

        # 5. 🧠 AI 추론 시뮬레이션 (The Creative Part!)
        # 데이터를 받아서 실제로 LLM(대형 언어 모델)에 전달할 형식의 프롬프트를 생성합니다.
        print("\n⭐ [AI 추론 시뮬레이션] : 이 정보를 LLM에게 전달하는 프롬프트 형식:")
        
        # 💡 이 부분이 가장 중요합니다! 데이터를 어떻게 포장하느냐가 AI의 성능을 좌우합니다.
        prompt = f"""
        --- 문제 풀이 요청 ---
        [과목]: {subject}
        [질문]: {question}
        [보기]:
        A. {choices_list[0]}
        B. {choices_list[1]}
        C. {choices_list[2]}
        D. {choices_list[3] if len(choices_list) > 3 else 'N/A'}
        ---
        지문과 보기를 참고하여 가장 정확한 답을 [A, B, C, D] 중 하나로만 선택하시오.
        """
        print("------------------------------------------------------------------")
        print(prompt)
        print("------------------------------------------------------------------")
        
        time.sleep(0.5) # 시각적 효과를 위해 잠시 대기합니다.

# ===============================================================================
# 🚀 메인 실행부
# ===============================================================================

if __name__ == "__main__":
    # 1. 데이터셋 로드 (안전 로직 적용)
    dataset_iterator = load_data_with_robustness(
        dataset_id=DATASET_NAME, 
        split_name='train',  # 'train' 스플릿을 기본으로 합니다.
        sample_limit=SAMPLE_COUNT
    )
    
    if dataset_iterator:
        # 2. 분석 실행
        run_mlmu_quiz_analysis(dataset_iterator)
    else:
        print("\n🔥 실습을 진행할 수 없습니다. 데이터 로드에 실패했습니다.")